# MAI-Transcribe-2

Transcribe noisy multilingual audio with diarization, timestamps, and domain biasing.

| | |
|---|---|
| Released | 2026-09-03 |
| Publisher | Microsoft AI |
| Capability | [Audio / Speech](../../3-primers/audio-speech.md) |
| Model card | [MAI-Transcribe-2](https://ai.azure.com/catalog/models/MAI-Transcribe-2) |
| Announcement | [Release post](https://microsoft.ai/news/mai-transcribe-2-is-the-fastest-most-accurate-and-cheapest-speech-recognition-model-in-the-world/) |
| Pricing | $0.10 per hour of audio for a limited time; standard pricing is $0.36 per hour — [source](https://microsoft.ai/models/mai-transcribe-2/) |

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

## About this release

> **Public preview.** MAI-Transcribe runs on the LLM Speech API, which is in public preview — no SLA, not recommended for production workloads.

### Before You Begin

| Detail | Value |
|---|---|
| Deployment regions | See [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech) — LLM Speech is available in a subset of Speech regions |

**Required environment variables:**

```text
MICROSOFT_FOUNDRY_ENDPOINT   # e.g. https://<resource>.services.ai.azure.com/api/projects/<project>
MICROSOFT_FOUNDRY_API_KEY    # AIServices (Speech) resource key
```

Optional overrides:

```text
AZURE_SPEECH_ENDPOINT        # use when your Speech resource differs from the Foundry project resource
```

> MAI-Transcribe-2 needs **no deployment name**. Set `enhancedMode.enabled` to `true`
> and `enhancedMode.model` to `MAI-Transcribe-2`. The enabled flag defaults to `false`;
> a model name alone does not enable MAI-Transcribe. Put timestamps and transcript style
> inside `enhancedMode.modelOptions`, as shown in the [Speech documentation](https://learn.microsoft.com/azure/ai-services/speech-service/mai-transcribe?pivots=programming-language-rest#use-a-mai-transcribe-model).

### What MAI-Transcribe-2 provides

MAI-Transcribe-2 is a multilingual speech recognition model for noisy and real-world audio.
It supports:

- **60 languages** with automatic language identification and code switching
- **Speaker diarization** to identify who spoke
- **Word-level timestamps** for alignment, search, navigation, and editing
- **Keyword biasing** through `phraseList` for specialized vocabulary
- **Configurable transcript style**: `verbatim` (the default) preserves fillers and false starts;
  `clean` removes them for readable captions and notes

The model is designed for contact-centre documentation, clinical notes, accessibility tools,
content creation, and voice-agent workflows.

### API surface

| Piece | Value |
|---|---|
| Endpoint | `https://<resource>.cognitiveservices.azure.com/speechtotext/transcriptions:transcribe?api-version=2025-10-15` |
| Body | `multipart/form-data` — `audio` file + `definition` JSON |
| Auth header | `Ocp-Apim-Subscription-Key`, or Entra ID via `DefaultAzureCredential` |
| Python SDK | [`azure-ai-transcription`](https://pypi.org/project/azure-ai-transcription/) — `TranscriptionClient.transcribe()` |

A request can combine model selection with diarization, word timestamps, transcript style,
and domain vocabulary:

```json
{
  "locales": ["en-US"],
  "phraseList": {
    "phrases": ["Microsoft Foundry", "AI", "DevOps"]
  },
  "diarization": {
    "enabled": true
  },
  "enhancedMode": {
    "enabled": true,
    "model": "MAI-Transcribe-2",
    "modelOptions": {
      "timestamps": "word",
      "transcribeStyle": "verbatim"
    }
  }
}
```

Omit `locales` to allow automatic language identification and code switching. When supplied,
provide only one language, such as `en-US`; this is a strong hint, not a list of candidates.

### What does a confidence value of zero mean?

[Issue #19](https://github.com/microsoft-foundry/model-releases/issues/19) reports zero confidence
values after enabling MAI-Transcribe. The request configuration above follows the Speech docs,
but those docs do not establish the meaning of these zero values. Zero alone does not prove
a failed request or an incorrect transcript. Keep enhanced mode enabled; do not turn it off
to obtain nonzero scores from a different transcription path.

We preserve confidence exactly as returned in the phrase table and raw JSON, but omit confidence
averages from the noise and model comparisons. Compare transcripts with a hand-verified reference
using word or character error rate (WER/CER) instead of treating confidence as an accuracy score.

### Know the boundaries

- Speaker diarization is currently limited to shorter recordings in preview. Longer requests
  can time out or fail with service errors; for long recordings, transcribe without diarization
  and use word timestamps with a separate diarization step.
- Audio and request limits, supported locales, and feature availability can change while the
  API is in preview. Check the live Speech documentation before building a production workflow.
- Diarization attributes words to speakers within the returned segments; inspect the response
  shape before coupling downstream logic to a particular segment layout.
- Pricing is time-sensitive. Treat the MAI model page as the source of truth for the current
  limited-time and standard rates.

## MAI-Transcribe-2: speaker-aware transcription

**Before we begin**

- Pricing: $0.10 per audio hour for a limited time; standard pricing is $0.36 per hour. Confirm current rates on the [model page](https://microsoft.ai/models/mai-transcribe-2/).
- Supported input formats: WAV, MP3, and FLAC, up to 300 MB.
- We need a Microsoft Foundry AI Services/Speech resource in a supported region. Run [setup.ipynb](../../setup.ipynb) first.
- Set `MICROSOFT_FOUNDRY_ENDPOINT` and optionally `MICROSOFT_FOUNDRY_API_KEY`. Use `AZURE_SPEECH_ENDPOINT` when Speech uses a separate resource.

In [1]:
%pip install azure-ai-transcription azure-identity python-dotenv pandas --quiet

import json
import os
import time
from collections.abc import Mapping
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from azure.ai.transcription import TranscriptionClient
from azure.ai.transcription.models import (
    EnhancedModeProperties,
    PhraseListProperties,
    TranscriptionContent,
    TranscriptionOptions,
)
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

foundry_endpoint = os.environ.get("MICROSOFT_FOUNDRY_ENDPOINT")
speech_endpoint = os.environ.get("AZURE_SPEECH_ENDPOINT")
if not (speech_endpoint or foundry_endpoint):
    raise EnvironmentError(
        "Set MICROSOFT_FOUNDRY_ENDPOINT or AZURE_SPEECH_ENDPOINT before continuing."
    )
if not speech_endpoint:
    parsed = urlparse(foundry_endpoint)
    speech_endpoint = f"{parsed.scheme}://{parsed.netloc}"

api_key = os.environ.get("AZURE_SPEECH_API_KEY") or os.environ.get(
    "MICROSOFT_FOUNDRY_API_KEY"
)
if api_key:
    credential = AzureKeyCredential(api_key)
    auth_method = "API key"
else:
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    auth_method = "Entra ID (DefaultAzureCredential)"

MODEL_15 = "mai-transcribe-1.5"
MODEL_2 = "MAI-Transcribe-2"

cwd = Path.cwd()
NOTEBOOK_DIR = cwd if cwd.name == "mai-transcribe-2" else cwd / "4-models/mai-transcribe-2"
DATA_DIR = NOTEBOOK_DIR.parents[1] / "0-archive" / "mai-transcribe-1.5" / "data"  # Shared with the archived 1.5 capsule
OUT_DIR = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(exist_ok=True)

client = TranscriptionClient(endpoint=speech_endpoint, credential=credential)
print(f"Endpoint: {speech_endpoint}")
print(f"Auth: {auth_method}")
print(f"Audio: {DATA_DIR.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")

Note: you may need to restart the kernel to use updated packages.
Endpoint: https://mai-model-releases-resource.cognitiveservices.azure.com/
Auth: API key
Audio: /workspaces/model-releases/0-archive/mai-transcribe-1.5/data
Output: /workspaces/model-releases/4-models/mai-transcribe-2/output


## 2. Configure MAI-Transcribe-1.5 and 2

Both versions use the same endpoint, credential, and SDK client. `make_options` explicitly enables `enhanced_mode` for both versions and nests MAI-Transcribe-2 options under its `modelOptions` wire field. These preview fields are set by wire name because SDK versions may not expose named properties for them.

For the version comparison, we deliberately omit model-exclusive features. This keeps the input audio, locale, and request shape equivalent so differences reflect the selected model rather than diarization or timestamp processing.

In [2]:
def value(item, *names, default=None):
    """Read a field from an SDK mapping or model object."""
    for name in names:
        if isinstance(item, Mapping) and name in item:
            return item[name]
        candidate = getattr(item, name, None)
        if candidate is not None:
            return candidate
    return default


def make_options(
    model_name,
    *,
    locales=None,
    phrases=None,
    biasing_weight=None,
    diarization=False,
    timestamps=None,
    transcribe_style=None,
):
    enhanced = EnhancedModeProperties()
    enhanced["enabled"] = True
    enhanced["model"] = model_name

    model_options = {}
    if timestamps:
        model_options["timestamps"] = timestamps
    if transcribe_style:
        model_options["transcribeStyle"] = transcribe_style
    if model_options:
        enhanced["modelOptions"] = model_options

    settings = {"enhanced_mode": enhanced}
    if locales:
        settings["locales"] = locales
    if phrases:
        settings["phrase_list"] = PhraseListProperties(
            phrases=phrases,
            biasing_weight=biasing_weight if biasing_weight is not None else 1.0,
        )

    options = TranscriptionOptions(**settings)
    if diarization:
        options["diarization"] = {"enabled": True}
    return options


def transcribe(audio_path, *, model_name=MODEL_2, **kwargs):
    with Path(audio_path).open("rb") as audio:
        content = TranscriptionContent(
            definition=make_options(model_name, **kwargs),
            audio=audio,
        )
        return client.transcribe(content)


def transcript_text(result):
    combined = value(result, "combined_phrases", "combinedPhrases", default=[])
    return value(combined[0], "text", default="") if combined else ""


def phrase_rows(result):
    rows = []
    for phrase in value(result, "phrases", default=[]) or []:
        offset = value(phrase, "offset_milliseconds", "offsetMilliseconds", default=0)
        duration = value(phrase, "duration_milliseconds", "durationMilliseconds", default=0)
        rows.append(
            {
                "speaker": value(phrase, "speaker", "speaker_id", "speakerId"),
                "start_seconds": round(offset / 1000, 3),
                "end_seconds": round((offset + duration) / 1000, 3),
                "locale": value(phrase, "locale"),
                "confidence": value(phrase, "confidence"),
                "text": value(phrase, "text", default=""),
            }
        )
    return rows


def word_rows(result):
    rows = []
    for phrase in value(result, "phrases", default=[]) or []:
        phrase_speaker = value(phrase, "speaker", "speaker_id", "speakerId")
        for word in value(phrase, "words", default=[]) or []:
            offset = value(word, "offset_milliseconds", "offsetMilliseconds", default=0)
            duration = value(word, "duration_milliseconds", "durationMilliseconds", default=0)
            rows.append(
                {
                    "speaker": value(word, "speaker", "speaker_id", "speakerId", default=phrase_speaker),
                    "word": value(word, "word", "text", default=""),
                    "start_seconds": round(offset / 1000, 3),
                    "end_seconds": round((offset + duration) / 1000, 3),
                }
            )
    return rows


def timed_transcription(audio_path, *, model_name=MODEL_2, **kwargs):
    started = time.perf_counter()
    result = transcribe(audio_path, model_name=model_name, **kwargs)
    elapsed = time.perf_counter() - started
    duration_ms = value(result, "duration_milliseconds", "durationMilliseconds", default=0)
    duration_seconds = duration_ms / 1000
    return result, elapsed, duration_seconds

## 3. Load and validate audio

We validate the input before spending a service call. The cell checks the path, extension, size, duration, sample rate, and channel count. Change `sample_audio` to another WAV, MP3, or FLAC fixture to test a different recording.

In [3]:
%pip install soundfile --quiet

import soundfile as sf

SUPPORTED_FORMATS = {".wav", ".mp3", ".flac"}
sample_audio = DATA_DIR / "normal - on call agent building.mp3"

if not sample_audio.is_file():
    raise FileNotFoundError(f"Audio fixture not found: {sample_audio.resolve()}")
if sample_audio.suffix.lower() not in SUPPORTED_FORMATS:
    raise ValueError(f"Expected one of {sorted(SUPPORTED_FORMATS)}, got {sample_audio.suffix}")
if sample_audio.stat().st_size > 300 * 1024 * 1024:
    raise ValueError("Audio exceeds the 300 MB request limit")

audio_info = sf.info(sample_audio)
audio_metadata = {
    "path": str(sample_audio.resolve()),
    "format": audio_info.format,
    "duration_seconds": round(audio_info.duration, 2),
    "sample_rate_hz": audio_info.samplerate,
    "channels": audio_info.channels,
    "size_mb": round(sample_audio.stat().st_size / (1024 * 1024), 2),
}
display(pd.DataFrame([audio_metadata]))

Note: you may need to restart the kernel to use updated packages.


,path,format,duration_seconds,sample_rate_hz,channels,size_mb
0,/workspaces/model-releases/0-archive/mai-trans...,MP3,49.82,48000,1,1.52


## 4. Call MAI-Transcribe-2

This first call requests both speaker diarization and word timestamps. Diarization is best suited to shorter recordings during preview; long recordings can time out. The response is kept intact for later export.

The next examples retain the useful 1.5 workflow: readable versus verbatim text, phrase-list biasing, and multilingual recognition.

In [4]:
mai2_result, mai2_latency, mai2_audio_seconds = timed_transcription(
    sample_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    diarization=True,
    timestamps="word",
)

print(transcript_text(mai2_result))
print(f"Audio: {mai2_audio_seconds:.2f}s | Latency: {mai2_latency:.2f}s")

Hello, I am building an on-call agent for DevOps, and the idea is to get engineers paged for in, when, okay, hear me out. When engineers are paged for incidents, and they technically spend like 10, 15 minutes understanding the context, checking dashboards, logs, deploys, and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and, yeah, does a blind, um, execution of something?
Audio: 49.82s | Latency: 0.74s


## 5. Extract speaker diarization and timestamps

Diarization answers **who spoke when**. Word timestamps answer **when each recognized token occurred**. We normalize both SDK object and mapping responses, sort by start time, and keep seconds numeric so downstream code can filter or seek accurately.

Speaker labels identify distinct voices within this recording; they do not identify real people.

In [5]:
diarized_columns = [
    "speaker",
    "start_seconds",
    "end_seconds",
    "locale",
    "confidence",
    "text",
]
word_columns = ["speaker", "word", "start_seconds", "end_seconds"]
diarized_df = pd.DataFrame(phrase_rows(mai2_result), columns=diarized_columns)
words_df = pd.DataFrame(word_rows(mai2_result), columns=word_columns)
if not diarized_df.empty:
    diarized_df = diarized_df.sort_values("start_seconds", ignore_index=True)
if not words_df.empty:
    words_df = words_df.sort_values("start_seconds", ignore_index=True)

print("Diarized segments")
display(diarized_df)
print("Word timestamps")
display(words_df.head(30))

Diarized segments


,speaker,start_seconds,end_seconds,locale,confidence,text
0,0,8.2,48.572,en,0,"Hello, I am building an on-call agent for DevO..."


Word timestamps


,speaker,word,start_seconds,end_seconds
0,0,"Hello,",8.20,8.800
1,0,I,9.36,9.440
2,0,am,9.56,9.739
3,0,building,9.88,10.380
4,0,an,10.92,11.040
5,0,on-call,11.20,11.759
6,0,agent,12.24,12.679
7,0,for,12.80,12.919
8,0,"DevOps,",13.04,13.559
9,0,and,14.04,14.219


### Retain the MAI-Transcribe-1.5 workflow

MAI-Transcribe-2 retains the practical controls demonstrated in the 1.5 notebook:

- Compare explicit `clean` and `verbatim` transcript styles on disfluent speech; the default is `verbatim`.
- Bias specialized vocabulary with a short `phraseList`.
- Let the model identify language and code switching automatically, or pin one locale such as `es-ES` when the language is known.

These are separate calls so each option's effect remains visible.

In [6]:
style_audio = DATA_DIR / "filler words for verbatim.mp3"
readable_result = transcribe(
    style_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    transcribe_style="clean",
)
verbatim_result = transcribe(
    style_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    transcribe_style="verbatim",
)

phrases = ["Microsoft Foundry", "AI", "Speech Recognition", "DevOps", "Copilot"]
bias_audio = DATA_DIR / "rain + specific content.mp3"
unbiased_result = transcribe(bias_audio, model_name=MODEL_2, locales=["en-US"])
biased_result = transcribe(
    bias_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    phrases=phrases,
    biasing_weight=1.5,
)

spanish_audio = DATA_DIR / "Multilingual Speech Transcript - Spanish.mp3"
auto_language_result = transcribe(spanish_audio, model_name=MODEL_2)
pinned_language_result = transcribe(
    spanish_audio,
    model_name=MODEL_2,
    locales=["es-ES"],
)
feature_examples = pd.DataFrame(
    [
        {"example": "clean", "transcript": transcript_text(readable_result)},
        {"example": "verbatim", "transcript": transcript_text(verbatim_result)},
        {"example": "without phrase list", "transcript": transcript_text(unbiased_result)},
        {"example": "with phrase list", "transcript": transcript_text(biased_result)},
        {"example": "Spanish auto-detect", "transcript": transcript_text(auto_language_result)},
        {"example": "Spanish pinned", "transcript": transcript_text(pinned_language_result)},
    ]
)
display(feature_examples)

,example,transcript
0,clean,"Hello, I am Bethany, and I wanted to share an ..."
1,verbatim,"Hello, um, I am Bethany, and um, I wanted to s..."
2,without phrase list,"Hello, my name is Bethany and I'm recording th..."
3,with phrase list,"Hello, my name is Bethany and I'm recording th..."
4,Spanish auto-detect,"Hola, me llamo Gustavo y estoy grabando este m..."
5,Spanish pinned,"Hola, me llamo Gustavo y estoy grabando este m..."


## 6. Transcribe controlled noise levels

This compact experiment brings the 1.5 notebook's controlled-noise approach to MAI-Transcribe-2. It mixes the clean fixture with reproducible white noise at 20, 10, and 0 dB signal-to-noise ratio (SNR), then transcribes each WAV with the same locale and request shape.

Each request is billed. This is a small diagnostic, not an accuracy benchmark: add a hand-verified reference transcript and WER/CER scoring before making quality claims.

In [7]:
import numpy as np


def mix_white_noise_at_snr(speech, snr_db, random_generator):
    noise = random_generator.standard_normal(len(speech))
    speech_power = float(np.mean(speech**2))
    noise_power = float(np.mean(noise**2))
    scale = np.sqrt(speech_power / (noise_power * 10 ** (snr_db / 10)))
    mixed = speech + scale * noise
    peak = float(np.max(np.abs(mixed)))
    return mixed / peak * 0.99 if peak > 1.0 else mixed


noise_source = DATA_DIR / "normal - on call agent building.mp3"
noise_output_dir = OUT_DIR / "noise-levels"
noise_output_dir.mkdir(exist_ok=True)
noise_samples, noise_sample_rate = sf.read(noise_source, dtype="float64", always_2d=True)
clean_speech = noise_samples.mean(axis=1)
noise_levels_db = [20, 10, 0]
random_generator = np.random.default_rng(20260915)
noise_records = []

for snr_db in noise_levels_db:
    mixed_audio = mix_white_noise_at_snr(clean_speech, snr_db, random_generator)
    mixed_path = noise_output_dir / f"white_{snr_db}dB.wav"
    sf.write(mixed_path, mixed_audio, noise_sample_rate, subtype="PCM_16")
    started = time.perf_counter()
    try:
        result = transcribe(mixed_path, model_name=MODEL_2, locales=["en-US"])
        latency = time.perf_counter() - started
        rows = phrase_rows(result)
        duration_seconds = value(result, "duration_milliseconds", "durationMilliseconds", default=0) / 1000
        noise_records.append(
            {
                "snr_db": snr_db,
                "audio": mixed_path.name,
                "detected_locales": ", ".join(sorted({row["locale"] for row in rows if row["locale"]})),
                "latency_seconds": round(latency, 2),
                "rtf": round(latency / duration_seconds, 3) if duration_seconds else None,
                "transcript": transcript_text(result),
                "error": None,
            }
        )
    except Exception as exc:
        noise_records.append(
            {
                "snr_db": snr_db,
                "audio": mixed_path.name,
                "detected_locales": "",
                "latency_seconds": round(time.perf_counter() - started, 2),
                "rtf": None,
                "transcript": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

noise_levels_df = pd.DataFrame(noise_records)
noise_levels_df.to_csv(OUT_DIR / "noise-level-transcription.csv", index=False)
display(noise_levels_df.drop(columns="transcript"))
for row in noise_records:
    print(f"\n--- white noise at {row['snr_db']} dB ---")
    print(row["error"] or row["transcript"])
print(f"Saved: {OUT_DIR / 'noise-level-transcription.csv'}")

,snr_db,audio,detected_locales,latency_seconds,rtf,error
0,20,white_20dB.wav,en,0.48,0.010,None
1,10,white_10dB.wav,en,0.41,0.008,None
2,0,white_0dB.wav,en,0.41,0.008,None



--- white noise at 20 dB ---
Hello, I am building an on-call agent for DevOps, and the idea is to get engineers paged for in, when, okay, hear me out. When engineers are paged for incidents, they technically spend like 10 to 15 minutes understanding the context, checking dashboards, logs, deploys, and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and, yeah, does a blind, um, execution of something?

--- white noise at 10 dB ---
Hello, I am building an on-call agent for DevOps, and the idea is to get engineers paid for in, when, okay, hear me out. When engineers are paid for incidents, they technically spend like 10, 15 minutes understanding the context, checking dashboards, logs, deploys, and related alerts before they can start diagnosing. What if you had an agent that automates that chaos process and, yeah, does a blind, um, execution of something?

--- white noise at 0 dB ---
Hello, I am building an on-call agent for De

## 7. Compare MAI-Transcribe-1.5 with 2

We send the same file with the same pinned locale to both models, with enhanced mode explicitly enabled for each. No 2-only option is enabled in these two calls. The comparison captures latency, real-time factor (RTF), transcript text, phrase timing coverage, speaker labels, detected locales, and errors. Confidence is preserved in the raw responses, not used as a model-quality metric.

This is a workload check, not a benchmark claim. Run several representative files and repeat calls before drawing conclusions about quality or speed.

In [8]:
comparison_records = []
comparison_results = {}

for model_name in (MODEL_15, MODEL_2):
    started = time.perf_counter()
    try:
        result, latency, audio_seconds = timed_transcription(
            sample_audio,
            model_name=model_name,
            locales=["en-US"],
        )
        comparison_results[model_name] = result
        phrases_for_model = phrase_rows(result)
        words_for_model = word_rows(result)
        comparison_records.append(
            {
                "model": model_name,
                "audio_seconds": round(audio_seconds, 2),
                "latency_seconds": round(latency, 2),
                "rtf": round(latency / audio_seconds, 3) if audio_seconds else None,
                "detected_locales": ", ".join(
                    sorted({row["locale"] for row in phrases_for_model if row["locale"]})
                ),
                "timestamped_phrases": sum(
                    row["end_seconds"] > row["start_seconds"] for row in phrases_for_model
                ),
                "timestamped_words": len(words_for_model),
                "speakers": ", ".join(
                    sorted({str(row["speaker"]) for row in phrases_for_model if row["speaker"] is not None})
                ),
                "transcript": transcript_text(result),
                "error": None,
            }
        )
    except Exception as exc:
        comparison_records.append(
            {
                "model": model_name,
                "audio_seconds": None,
                "latency_seconds": round(time.perf_counter() - started, 2),
                "rtf": None,
                "detected_locales": "",
                "timestamped_phrases": 0,
                "timestamped_words": 0,
                "speakers": "",
                "transcript": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

comparison_df = pd.DataFrame(comparison_records)
display(comparison_df.drop(columns="transcript"))
for row in comparison_records:
    print(f"\n--- {row['model']} ---")
    print(row["error"] or row["transcript"])

comparison_df.to_csv(OUT_DIR / "mai-transcribe-1.5-vs-2.csv", index=False)

,model,audio_seconds,latency_seconds,rtf,detected_locales,timestamped_phrases,timestamped_words,speakers,error
0,mai-transcribe-1.5,49.82,1.28,0.026,en,1,0,,None
1,MAI-Transcribe-2,49.82,0.46,0.009,en,1,0,,None



--- mai-transcribe-1.5 ---
Hello, I am building an on-call agent for DevOps, and the idea is to get engineers paid for in- uh, when... Okay, hear me out. When engineers are paid for incidents, in the technically spend like 10-15 minutes understanding the context, checking dashboards, logs, deploys, and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and, yeah, does a blind, um, execution of something?

--- MAI-Transcribe-2 ---
Hello, I am building an on-call agent for DevOps, and the idea is to get engineers paged for in, when, okay, hear me out. When engineers are paged for incidents, and they technically spend like 10, 15 minutes understanding the context, checking dashboards, logs, deploys, and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and, yeah, does a blind, um, execution of something?


## 8. Display and export results

We export machine-readable artifacts for downstream review:

- The raw MAI-Transcribe-2 response as JSON
- Timestamped words as CSV
- Diarized segments as CSV
- Side-by-side comparison metrics and transcripts as CSV

The raw response preserves preview fields that our normalized tables may not yet expose.

In [9]:
def jsonable(item):
    if hasattr(item, "as_dict"):
        return jsonable(item.as_dict())
    if isinstance(item, Mapping):
        return {str(key): jsonable(val) for key, val in item.items()}
    if isinstance(item, (list, tuple)):
        return [jsonable(entry) for entry in item]
    return item


raw_path = OUT_DIR / "mai-transcribe-2-response.json"
raw_path.write_text(
    json.dumps(jsonable(mai2_result), indent=2, default=str),
    encoding="utf-8",
)
diarized_path = OUT_DIR / "mai-transcribe-2-diarized.csv"
words_path = OUT_DIR / "mai-transcribe-2-word-timestamps.csv"
comparison_path = OUT_DIR / "mai-transcribe-1.5-vs-2.csv"

diarized_df.to_csv(diarized_path, index=False)
words_df.to_csv(words_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

for path in (raw_path, diarized_path, words_path, comparison_path):
    print(f"Saved: {path}")

Saved: /workspaces/model-releases/4-models/mai-transcribe-2/output/mai-transcribe-2-response.json
Saved: /workspaces/model-releases/4-models/mai-transcribe-2/output/mai-transcribe-2-diarized.csv
Saved: /workspaces/model-releases/4-models/mai-transcribe-2/output/mai-transcribe-2-word-timestamps.csv
Saved: /workspaces/model-releases/4-models/mai-transcribe-2/output/mai-transcribe-1.5-vs-2.csv


### Your Turn to Explore

1. Replace `sample_audio` with a short meeting recording and compare the returned speaker count with the voices we hear.
2. Generate captions from `words_df`, grouping words into readable time windows.
3. Run the comparison over several noisy and multilingual clips, then summarize median RTF and review transcript differences against hand-verified references.

### Summary

We reused the MAI-Transcribe-1.5 client pattern with MAI-Transcribe-2, validated audio before submission, requested speaker diarization and word timestamps, and normalized both into tables. The feature examples cover transcript style, phrase-list biasing, automatic and multi-locale multilingual recognition, and reproducible transcription at controlled noise levels. We also compared 1.5 and 2 under equivalent baseline settings and exported the results for review.

Reach for MAI-Transcribe-2 when speaker turns or precise alignment matter alongside multilingual recognition, domain vocabulary, and noisy audio. Because the service is in preview, validate feature limits, response shape, accuracy, latency, and current pricing against our own workload before production use.

See the [Audio / Speech primer](../../3-primers/audio-speech.md) for related concepts.

### References

- [MAI-Transcribe-2 model page](https://microsoft.ai/models/mai-transcribe-2/) - features, language coverage, benchmarks, and current pricing.
- [MAI-Transcribe-2 announcement](https://microsoft.ai/news/mai-transcribe-2-is-the-fastest-most-accurate-and-cheapest-speech-recognition-model-in-the-world/) - release context and comparative results.
- [MAI-Transcribe in Azure Speech](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry) - request parameters, diarization, timestamps, styles, and limits.
- [LLM Speech transcription and translation](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/llm-speech?tabs=new-foundry%2Cwindows&pivots=programming-language-python) - authentication, Python SDK, and response handling.
- [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech) - current endpoint and regional availability guidance.